In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from meteor import MeteorInterface

In [ ]:
# FASTMIP Phase 1:
# METEOR emulation of annual tas and pr timeseries for all ESMs and 5 SSP scenarios.
# 10 ensemble members, both with native METEOR noise and scaled to FAIR GSAT 50th percentile timeseries.


In [3]:
# FASTMIP Phase 1 output to generate:
# Up to 2100
# 2.5x2.5 degree common grid
# 10 member ensemble for each ESM and scenario
# Annual mean tas and pr (gridded)
# Both METEOR raw output and scaled to provided GSAT timeseries

ESMs = ['ACCESS-ESM1-5']#, 'CMCC-CM2-SR5', 'CNRM-CM6-1', 'CanESM5', 'EC-Earth3', 'INM-CM5-0', 'IPSL-CM6A-LR', 'MIROC-ES2L', 'MIROC6', 'MPI-ESM1-2-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0']
scenarios = ['ssp119']#, 'ssp126', 'ssp245', 'ssp370', 'ssp585']
FAIRpercentiles = ['5', '10', '50', '90', '95']
start_year = 2020
end_year = 2100
n_members = 10
output_dir = '../data/FASTMIP_phase1/METEOR_emulations/'
scaling_dir = '../data/FASTMIP_phase1/FAIR_GSAT/'

def format_FAIRssp(scenario):
    return f"SSP{scenario[3]}-{scenario[4:]}"

In [8]:
# Create emulators for list of ESMs (uses cached models if available)
for esm in ESMs:
    print(f"Creating emulator for {esm}...")
    emulator = MeteorInterface(
        model=esm,
        variables=['tas', 'pr'],
        cache_dir='../cache'
    )
    print(f"Emulator created for {emulator.model} with variables: {emulator.variables}")
    
    print(f"Training emulator for {emulator.model}...")
    emulator.train(verbose=True)

    for scenario in scenarios:
        print(f"Emulating scenario {scenario}...")

        # Emulate gridded outputs for each scenario and save:
        ensemble_gridded = emulator.generate_ensemble_outputs(
            scenario=scenario,
            start_year=start_year,
            end_year=end_year,
            n_realizations=n_members,
            timeseries=['global'], 
            gridded={
                'annual': list(range(start_year, end_year + 1)),  # Annual mean grids for all years 
                #'monthly': list(range(start_year, end_year + 1)),  # only annual currently.
            },
            save_to=f"{output_dir}METEOR_{esm}_{scenario}.nc"
        )

        # Scale output to FAIR GSAT:
        # (Could also loop through percentiles here, for now only using 50th)
        for percentile in ['50']: #FAIRpercentiles:    
            scaling_ts = xr.open_dataset(f"{scaling_dir}GSAT_SSPmarker_{format_FAIRssp(scenario)}.nc")['tas'].sel(percentile=percentile, SCM='FaIRv1.6.2', time=slice(start_year, end_year)).rename({'time': 'year'})
            #scaling_ts['year'] = pd.to_datetime(scaling_ts['year'].values.astype(str), format='%Y')

            ensemble_scaled = emulator.generate_ensemble_outputs(
                scenario=scenario,
                start_year=start_year,
                end_year=end_year,
                n_realizations=n_members,
                timeseries=['global'], 
                gridded={
                    'annual': list(range(start_year, end_year + 1)),  # Annual mean grids for all years
                    #'monthly': list(range(start_year, end_year + 1)),  # only annual currently.
                },
                temp_scaling_ts=scaling_ts,
                save_to=f"{output_dir}METEOR_{esm}_{scenario}_scaledtoFAIR_{percentile}percentile.nc"
            )   
        

Creating emulator for ACCESS-ESM1-5...
Emulator created for ACCESS-ESM1-5 with variables: ['tas', 'pr']
Training emulator for ACCESS-ESM1-5...
Training METEOR emulator for ACCESS-ESM1-5
Variables: tas, pr

🔧 Training TAS...
   → Training pattern scaling model...
      ⚠️  Cache miss: Model name mismatch: expected 'cmip6-ACCESS-ESM1-5-aer', found 'cmip6-ACCESS-ESM1-5-aer-tas'
🔧 Preparing pattern scaling training data for ACCESS-ESM1-5...
   ✅ Training data prepared for experiments:  ['base', 'co2x4', 'ssp245', 'sulxanom']
📥 Loading CICERO-SCM forcing data for ssp245...
   ✅ Loaded 801 concentration records
   ✅ Loaded 351 emission records
   ✅ Config: 1750-2100, emissions start: 1850
📦 Loading cached pattern scaling model from ../cache/pattern_scaling/cmip6-ACCESS-ESM1-5-aer-tas_pattern_scaling.pkl
✅ Pattern scaling model loaded from ../cache/pattern_scaling/cmip6-ACCESS-ESM1-5-aer-tas_pattern_scaling.pkl
   → Training noise model...
Model loaded from ../cache/noise_models/ACCESS-ESM1-5

TypeError: argument of type 'numpy.datetime64' is not iterable

In [9]:
# In a new cell after the error:
debug_frames

{'0_<module>': {'__name__': '__main__',
  '__doc__': 'Automatically created module for IPython interactive environment',
  '__package__': None,
  '__loader__': None,
  '__spec__': None,
  '__builtin__': <module 'builtins' (built-in)>,
  '__builtins__': <module 'builtins' (built-in)>,
  '_ih': ['',
   'import numpy as np\nimport xarray as xr\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom meteor import MeteorInterface',
   '# FASTMIP Phase 1 output to generate:\n# Up to 2100\n# 2.5x2.5 degree common grid\n# 10 member ensemble for each ESM and scenario\n# Annual mean tas and pr (gridded)\n# Both METEOR raw output and scaled to provided GSAT timeseries\n\nESMs = [\'ACCESS-ESM1-5\']#, \'CMCC-CM2-SR5\', \'CNRM-CM6-1\', \'CanESM5\', \'EC-Earth3\', \'INM-CM5-0\', \'IPSL-CM6A-LR\', \'MIROC-ES2L\', \'MIROC6\', \'MPI-ESM1-2-LR\', \'MPI-ESM1-2-HR\', \'MRI-ESM2-0\']\nscenarios = [\'ssp119\']#, \'ssp126\', \'ssp245\', \'ssp370\', \'ssp585\']\nFAIRpercentiles = [\'5\', \'10\', \'50\', \'